# Stock Price Prediction

**Objective:** Build a model to predict future stock prices based on historical data, including features like opening price, closing price, high, low, and trading volume.

**Workflow:**
1. Load historical stock data (AAPL via yfinance)
2. Explore and visualize the data
3. Preprocess: scale features, create sliding-window sequences
4. Train **Linear Regression** (baseline)
5. Train **LSTM** (PyTorch deep learning model)
6. Evaluate and compare both models
7. Visualize predictions vs actual prices

---
## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {device}')
print('All libraries imported successfully!')

---
## 2. Load Stock Data

Download Apple (AAPL) historical data from Yahoo Finance using `yfinance`.

In [ ]:
# Configuration
TICKER = 'AAPL'
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'
DATA_DIR = os.path.join(os.getcwd(), 'data')
SEQ_LENGTH = 10     # Number of past days used as input
TRAIN_RATIO = 0.8

FEATURE_COLS = ['Open', 'High', 'Low', 'Close', 'Volume']
TARGET_COL = 'Close'

os.makedirs(DATA_DIR, exist_ok=True)

csv_path = os.path.join(DATA_DIR, f'{TICKER}_{START_DATE}_{END_DATE}.csv')

# Download if not cached
if not os.path.exists(csv_path):
    print(f'Downloading {TICKER} data ({START_DATE} to {END_DATE}) ...')
    stock_df = yf.download(TICKER, start=START_DATE, end=END_DATE)
    stock_df.to_csv(csv_path)
    print(f'Saved to {csv_path}')
else:
    print(f'Using cached data: {csv_path}')

# Read and clean the CSV (handle multi-header yfinance format)
with open(csv_path, 'r') as f:
    first_lines = [f.readline() for _ in range(3)]

is_multi = any('Ticker' in line or 'Price' in line for line in first_lines)

if is_multi:
    stock_df = pd.read_csv(csv_path, header=[0, 1, 2], index_col=0)
    stock_df.columns = [col[0] for col in stock_df.columns]
    stock_df.index.name = 'Date'
else:
    stock_df = pd.read_csv(csv_path, index_col=0)

stock_df.columns = [col.strip().capitalize() if isinstance(col, str) else col for col in stock_df.columns]
stock_df.index = pd.to_datetime(stock_df.index)
stock_df.sort_index(inplace=True)

# Convert to numeric and drop NaN
for col in FEATURE_COLS:
    stock_df[col] = pd.to_numeric(stock_df[col], errors='coerce')
stock_df.dropna(inplace=True)

print(f'\nDataset shape: {stock_df.shape}')
print(f'Date range: {stock_df.index.min().date()} to {stock_df.index.max().date()}')
stock_df.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('=== Dataset Statistics ===')
stock_df[FEATURE_COLS].describe()

In [ ]:
# Price history plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

# OHLC prices
axes[0].plot(stock_df.index, stock_df['Close'], label='Close', color='#2563eb', lw=1.5)
axes[0].plot(stock_df.index, stock_df['Open'], label='Open', color='#64748b', lw=0.8, alpha=0.6)
axes[0].fill_between(stock_df.index, stock_df['Low'], stock_df['High'], alpha=0.15, color='steelblue', label='High-Low Range')
axes[0].set_title(f'{TICKER} Stock Price History', fontsize=14)
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Volume
axes[1].bar(stock_df.index, stock_df['Volume'], color='#64748b', alpha=0.6, width=1)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
corr_matrix = stock_df[FEATURE_COLS].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.3f', square=True)
plt.title(f'{TICKER} Feature Correlation Matrix')
plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing

1. **Scale** all features using MinMaxScaler
2. **Create sliding-window sequences** of length `SEQ_LENGTH`
3. **Split** chronologically into 80% train / 20% test

In [ ]:
# Scale features and target
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

data = stock_df[FEATURE_COLS].values
close_prices = np.array(stock_df[TARGET_COL].values, dtype=np.float64).reshape(-1, 1)

scaled_data = feature_scaler.fit_transform(data)
target_scaler.fit(close_prices)

# Create sliding-window sequences
target_idx = FEATURE_COLS.index(TARGET_COL)

X_seq, y_seq = [], []
for i in range(len(scaled_data) - SEQ_LENGTH):
    X_seq.append(scaled_data[i : i + SEQ_LENGTH])
    y_seq.append(scaled_data[i + SEQ_LENGTH, target_idx])  # next-day Close

X_seq = np.array(X_seq, dtype=np.float64)
y_seq = np.array(y_seq, dtype=np.float64).reshape(-1, 1)

# Chronological split
split_idx = int(len(X_seq) * TRAIN_RATIO)

dates = stock_df.index[SEQ_LENGTH:]
train_dates = dates[:split_idx]
test_dates = dates[split_idx:]

X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

print(f'Sequence length : {SEQ_LENGTH} days')
print(f'Total sequences : {len(X_seq)}')
print(f'Train set       : X={X_train.shape}, y={y_train.shape}')
print(f'Test set        : X={X_test.shape}, y={y_test.shape}')
print(f'Train period    : {train_dates[0].date()} to {train_dates[-1].date()}')
print(f'Test period     : {test_dates[0].date()} to {test_dates[-1].date()}')

---
## 5. Model 1: Linear Regression (Baseline)

Flatten each sequence window into a single feature vector and train a standard Linear Regression.

In [ ]:
# Flatten sequences for Linear Regression
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

lr_model = LinearRegression()
lr_model.fit(X_train_flat, y_train.ravel())

# Predictions (scaled)
lr_train_pred_s = lr_model.predict(X_train_flat).reshape(-1, 1)
lr_test_pred_s = lr_model.predict(X_test_flat).reshape(-1, 1)

# Inverse-transform to original price scale
lr_train_pred = target_scaler.inverse_transform(lr_train_pred_s)
lr_test_pred = target_scaler.inverse_transform(lr_test_pred_s)
y_train_orig = target_scaler.inverse_transform(y_train)
y_test_orig = target_scaler.inverse_transform(y_test)

# Metrics
def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE_%': mape, 'R2': r2}

lr_metrics = compute_metrics(y_test_orig, lr_test_pred)

print('Linear Regression - Test Metrics')
print('=' * 35)
for k, v in lr_metrics.items():
    print(f'  {k:8s}: {v:.4f}')

---
## 6. Model 2: LSTM (PyTorch)

A stacked LSTM network that processes the sequential windows natively.

In [ ]:
# LSTM Hyperparameters
HIDDEN_DIM = 64
NUM_LAYERS = 2
LEARNING_RATE = 0.001
EPOCHS = 20
BATCH_SIZE = 32


class StockLSTM(nn.Module):
    """Stacked LSTM regressor for time-series prediction."""

    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])


print(f'LSTM Configuration:')
print(f'  Hidden dim  : {HIDDEN_DIM}')
print(f'  Num layers  : {NUM_LAYERS}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  Epochs      : {EPOCHS}')
print(f'  Batch size  : {BATCH_SIZE}')

In [ ]:
# Prepare PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# Initialize model
input_dim = X_train.shape[2]
lstm_model = StockLSTM(input_dim, HIDDEN_DIM, NUM_LAYERS).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)

# Training loop
print('Training LSTM...')
print('-' * 40)

epoch_losses = []
lstm_model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = lstm_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * bx.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    epoch_losses.append(avg_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'  Epoch [{epoch + 1:3d}/{EPOCHS}]  Loss: {avg_loss:.6f}')

print('-' * 40)
print(f'Final training loss: {epoch_losses[-1]:.6f}')

In [ ]:
# Training loss curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), epoch_losses, color='#ea580c', lw=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('LSTM Training Loss Curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# LSTM Predictions
lstm_model.eval()
with torch.no_grad():
    lstm_train_pred_s = lstm_model(X_train_t).cpu().numpy()
    lstm_test_pred_s = lstm_model(X_test_t).cpu().numpy()

# Inverse-transform
lstm_train_pred = target_scaler.inverse_transform(lstm_train_pred_s)
lstm_test_pred = target_scaler.inverse_transform(lstm_test_pred_s)

# Metrics
lstm_metrics = compute_metrics(y_test_orig, lstm_test_pred)

print('LSTM - Test Metrics')
print('=' * 35)
for k, v in lstm_metrics.items():
    print(f'  {k:8s}: {v:.4f}')

---
## 7. Model Comparison

In [ ]:
# Comparison table
comparison_df = pd.DataFrame({
    'Linear Regression': lr_metrics,
    'LSTM': lstm_metrics,
}).T

comparison_df.style.highlight_min(
    subset=['MSE', 'RMSE', 'MAE', 'MAPE_%'], axis=0, color='lightgreen'
).highlight_max(
    subset=['R2'], axis=0, color='lightgreen'
).format('{:.4f}')

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_plot = ['RMSE', 'MAE', 'R2']
colors = ['#2563eb', '#f59e0b']

for ax, metric in zip(axes, metrics_to_plot):
    values = [lr_metrics[metric], lstm_metrics[metric]]
    bars = ax.bar(['Linear Regression', 'LSTM'], values, color=colors, alpha=0.85)
    ax.set_title(metric, fontsize=14)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * max(values),
                f'{val:.4f}', ha='center', fontsize=11)

plt.suptitle('Model Performance Comparison', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Prediction Visualization

In [ ]:
# Test set: Actual vs Predictions
plt.figure(figsize=(16, 7))

plt.plot(test_dates, y_test_orig.ravel(), label='Actual Close', color='black', lw=1.5, alpha=0.85)
plt.plot(test_dates, lr_test_pred.ravel(), label='Linear Regression', color='#2563eb', ls='--', alpha=0.75, lw=1.5)
plt.plot(test_dates, lstm_test_pred.ravel(), label='LSTM', color='#f59e0b', alpha=0.75, lw=1.5)

plt.title(f'{TICKER} Stock Price Prediction - Test Set', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Full timeline: Train + Test predictions
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Linear Regression
axes[0].plot(train_dates, y_train_orig.ravel(), color='#64748b', alpha=0.5, label='Train Actual')
axes[0].plot(train_dates, lr_train_pred.ravel(), color='#93c5fd', alpha=0.5, label='Train Pred')
axes[0].plot(test_dates, y_test_orig.ravel(), color='black', lw=1.5, label='Test Actual')
axes[0].plot(test_dates, lr_test_pred.ravel(), color='#2563eb', ls='--', lw=1.5, label='Test Pred')
axes[0].axvline(test_dates[0], color='red', ls=':', alpha=0.5, label='Train/Test Split')
axes[0].set_title('Linear Regression - Full Timeline', fontsize=13)
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left')
axes[0].grid(alpha=0.3)

# LSTM
axes[1].plot(train_dates, y_train_orig.ravel(), color='#64748b', alpha=0.5, label='Train Actual')
axes[1].plot(train_dates, lstm_train_pred.ravel(), color='#fcd34d', alpha=0.5, label='Train Pred')
axes[1].plot(test_dates, y_test_orig.ravel(), color='black', lw=1.5, label='Test Actual')
axes[1].plot(test_dates, lstm_test_pred.ravel(), color='#f59e0b', lw=1.5, label='Test Pred')
axes[1].axvline(test_dates[0], color='red', ls=':', alpha=0.5, label='Train/Test Split')
axes[1].set_title('LSTM - Full Timeline', fontsize=13)
axes[1].set_ylabel('Price ($)')
axes[1].set_xlabel('Date')
axes[1].legend(loc='upper left')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Prediction error distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lr_errors = (y_test_orig - lr_test_pred).ravel()
lstm_errors = (y_test_orig - lstm_test_pred).ravel()

axes[0].hist(lr_errors, bins=30, color='#2563eb', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='red', ls='--', alpha=0.5)
axes[0].set_title(f'Linear Regression Error Distribution\nMean: {lr_errors.mean():.2f}, Std: {lr_errors.std():.2f}')
axes[0].set_xlabel('Prediction Error ($)')
axes[0].set_ylabel('Frequency')

axes[1].hist(lstm_errors, bins=30, color='#f59e0b', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', ls='--', alpha=0.5)
axes[1].set_title(f'LSTM Error Distribution\nMean: {lstm_errors.mean():.2f}, Std: {lstm_errors.std():.2f}')
axes[1].set_xlabel('Prediction Error ($)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

---
## Summary

- Loaded historical stock data for AAPL (Jan 2020 - Jan 2023)
- Preprocessed: MinMaxScaler normalization + 10-day sliding-window sequences
- Trained and compared two models:
  - **Linear Regression** - Fast baseline using flattened sequence windows
  - **LSTM** - Deep learning model processing sequences natively
- Both models were evaluated using MSE, RMSE, MAE, MAPE, and R2
- Key insights:
  - Linear Regression provides a strong baseline for short-term price prediction
  - LSTM can capture temporal dependencies but requires more tuning and data
  - Stock price prediction is inherently challenging due to market volatility